# DATA-445: In-Class Assignment 11

## Dataset Overview

Consider the `framingham.csv` data file. The dataset is publicly available on the Kaggle website, and it is from an ongoing cardiovascular study on residents of the town of Framingham, Massachusetts. The classification goal is to predict whether the patient has 10-year risk of future coronary heart disease (CHD). The dataset provides the patients' information. It includes over 4,000 records and 15 attributes. Each attribute is a potential risk factor. There are both demographic, behavioral and medical risk factors.

### Data Dictionary

**Demographic:**
- **Sex:** male or female (Nominal)
- **Age:** Age of the patient (Continuous)

**Behavioral:**
- **Current Smoker:** whether or not the patient is a current smoker (Nominal)
- **Cigs Per Day:** the number of cigarettes that the person smoked on average in one day (Continuous)

**Medical (history):**
- **BP Meds:** whether or not the patient was on blood pressure medication (Nominal)
- **Prevalent Stroke:** whether or not the patient had previously had a stroke (Nominal)
- **Prevalent Hyp:** whether or not the patient was hypertensive (Nominal)
- **Diabetes:** whether or not the patient had diabetes (Nominal)

**Medical (current):**
- **Tot Chol:** total cholesterol level (Continuous)
- **Sys BP:** systolic blood pressure (Continuous)
- **Dia BP:** diastolic blood pressure (Continuous)
- **BMI:** Body Mass Index (Continuous)
- **Heart Rate:** heart rate (Continuous)
- **Glucose:** glucose level (Continuous)

**Target Variable:**
- **TenYearCHD:** 10 year risk of coronary heart disease CHD (binary: '1' = Yes, '0' = No)

## Exercise 1: Load the Data (3 points)

Using the pandas library, read the csv data file and create a data-frame called `heart`.

In [1]:
import pandas as pd

heart = pd.read_csv('framingham.csv')
heart.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


## Exercise 2: Handle Missing Values (3 points)

Remove observations with missing values.

In [2]:
heart = heart.dropna().reset_index(drop=True)
heart.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


## Exercise 3: Set Up Cross-Validation (4 points)

Using `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` as the predictor variables, and `TenYearCHD` is the target variable, run a five-fold cross validation experiment (taking into account the proportion of 0s and 1s).

In [3]:
from sklearn.model_selection import KFold

X = heart[["age", "currentSmoker", "totChol", "heartRate"]].copy()
y = heart["TenYearCHD"].copy()

skf = KFold(n_splits=5, shuffle=True, random_state=42)

## Exercise 4: Random Forest Model (8 points)

Using the train data-frame, build a random forest model (with 500 trees and max tree depth equal 5) in which: `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` are the predictor variables, and `TenYearCHD` is the target variable. 

Using this model, predict the risk of coronary disease of the patients in the test data-frame. Using 10% as cutoff value, report the average recall.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score

rf_md = RandomForestClassifier(n_estimators=500,
                               max_depth=5,
                               random_state=42,
                               n_jobs=-1)

# Running cross-validation 
rf_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    rf_md.fit(X_train, y_train)
    y_pred = rf_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred > 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    rf_recalls.append(score)
    print(f"Fold {i+1} Recall: {score}")

print(f"Average Recall: {sum(rf_recalls)/len(rf_recalls)}")

Fold 1 Recall: 0.7786885245901639
Fold 2 Recall: 0.897196261682243
Fold 3 Recall: 0.8421052631578947
Fold 4 Recall: 0.8454545454545455
Fold 5 Recall: 0.7884615384615384
Average Recall: 0.8303812266692772


## Exercise 5: Extra Trees Model (8 points)

Using the train data-frame, build an extra trees model (with 500 trees and max tree depth equal 5) in which: `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` are the predictor variables, and `TenYearCHD` is the target variable. 

Using this model, predict the risk of coronary disease of the patients in the test data-frame. Using 10% as cutoff value, report the average recall.

In [5]:
from sklearn.ensemble import ExtraTreesClassifier

et_md = ExtraTreesClassifier(n_estimators=500,
                             max_depth=5,
                             random_state=42,
                             n_jobs=-1)

et_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    et_md.fit(X_train, y_train)
    y_pred = et_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred > 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    et_recalls.append(score)
    print(f"Fold {i+1} Recall: {score}")

print(f"Average Recall: {sum(et_recalls)/len(et_recalls)}")

Fold 1 Recall: 0.8688524590163934
Fold 2 Recall: 0.9719626168224299
Fold 3 Recall: 0.9210526315789473
Fold 4 Recall: 0.9727272727272728
Fold 5 Recall: 0.8942307692307693
Average Recall: 0.9257651498751626


## Exercise 6: Model Comparison (3 points)

Using the results from parts (4) and (5), what model would you use to predict `TenYearCHD` based on the recall?

**Your Answer:**

Based on the results i would use the ExtraTreesClassifier model because it has the larger recall score.